Notebook: 04_twave_analysis.ipynb

Purpose: Characterize beat-level T-wave morphology and ambiguity.

Inputs:
- raw ECG waveforms
- delineated T-wave boundaries

Outputs:
- twave_features.parquet

# 04 — T-Wave Morphology Analysis

Extract beat-level T-wave metrics that capture amplitude, width, slope, symmetry, and ambiguity.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from ecg_analytics.datasets import PTBXLDataset, LUDBDataset, NSTDBDataset, INCARTDataset, QTDBDataset

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
artifacts_dir.mkdir(exist_ok=True)
data_dir = root_dir / 'data'

adapters = {
    'ptbxl': PTBXLDataset,
    'ludb': LUDBDataset,
    'nstdb': NSTDBDataset,
    'incart': INCARTDataset,
    'qtdb': QTDBDataset,
}

rows = []

for dataset_name, adapter_cls in adapters.items():
    dataset_path = data_dir / dataset_name
    if not dataset_path.exists():
        continue
    adapter = adapter_cls(data_dir=data_dir)
    try:
        record_names = adapter.list_records()
    except Exception:
        continue

    for record_name in record_names:
        try:
            record = adapter.load_record(record_name)
        except Exception:
            continue

        record_id = f'{dataset_name}/{record_name}'
        signal = np.asarray(record.signal, dtype=float)
        if signal.ndim == 1:
            signal = signal.reshape(-1, 1)
        annotations = getattr(record, 'annotations', []) or []
        lead_names = getattr(record, 'lead_names', []) or []

        for lead_index, lead_name in enumerate(lead_names):
            lead_signal = signal[:, lead_index] if signal.shape[1] > lead_index else signal[:, 0]
            t_points = [ann.sample for ann in annotations if getattr(ann, 'lead', None) == lead_index and getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() == 't']
            t_starts = [ann.sample for ann in annotations if getattr(ann, 'lead', None) == lead_index and getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() in {'(',} and 't' in getattr(getattr(ann, 'label', None), 'lower', lambda: '')()]
            t_ends = [ann.sample for ann in annotations if getattr(ann, 'lead', None) == lead_index and getattr(getattr(ann, 'symbol', None), 'lower', lambda: '')() == ')']
            beat_count = min(len(t_starts), len(t_points), len(t_ends))
            for beat_idx in range(beat_count):
                start = t_starts[beat_idx]
                peak = t_points[beat_idx]
                end = t_ends[beat_idx]
                segment = lead_signal[start:end] if end > start else np.array([])
                if segment.size < 2:
                    amplitude = np.nan
                    width = np.nan
                    slope = np.nan
                    symmetry = np.nan
                    biphasic = False
                    flattened = False
                    cluster = 'unknown'
                    ambiguity = np.nan
                    confidence = np.nan
                else:
                    amplitude = float(np.max(segment) - np.min(segment))
                    width = float((end - start) / float(getattr(record, 'fs', np.nan)) * 1000.0)
                    slope = float(np.nanmax(np.abs(np.diff(segment))) * float(getattr(record, 'fs', np.nan)))
                    rising = float(peak - start)
                    falling = float(end - peak) if end > peak else 1.0
                    symmetry = float(rising / falling) if falling > 0 else np.nan
                    biphasic = bool(np.any(segment < 0) and np.any(segment > 0))
                    flattened = bool(np.ptp(segment) < 0.1 * np.nanstd(lead_signal)) if np.nanstd(lead_signal) > 0 else False
                    cluster = 'biphasic' if biphasic else 'flattened' if flattened else 'monophasic'
                    ambiguity = float(np.std(segment) / (np.max(np.abs(segment)) + 1e-12))
                    confidence = float(np.clip(1.0 - ambiguity, 0.0, 1.0))

                rows.append({
                    'record_id': record_id,
                    'lead_id': str(lead_name).lower(),
                    'beat_id': int(beat_idx + 1),
                    't_amplitude_mv': amplitude,
                    't_width_ms': width,
                    't_slope': slope,
                    't_symmetry': symmetry,
                    'biphasic_flag': bool(biphasic),
                    'flattened_flag': bool(flattened),
                    'morphology_cluster': cluster,
                    't_end_ambiguity_score': ambiguity,
                    'morphology_confidence': confidence,
                })

if rows:
    twave = pd.DataFrame(rows)
else:
    twave = pd.DataFrame(columns=[
        'record_id','lead_id','beat_id','t_amplitude_mv','t_width_ms','t_slope','t_symmetry',
        'biphasic_flag','flattened_flag','morphology_cluster','t_end_ambiguity_score','morphology_confidence',
    ])

expected = {
    'record_id','lead_id','beat_id','t_amplitude_mv','t_width_ms','t_slope','t_symmetry',
    'biphasic_flag','flattened_flag','morphology_cluster','t_end_ambiguity_score','morphology_confidence',
}
assert expected.issubset(set(twave.columns)), 'twave schema mismatch'
assert twave[['record_id','lead_id','beat_id']].duplicated().sum() == 0

twave.to_parquet(artifacts_dir / 'twave_features.parquet', index=False)
print('Wrote twave_features.parquet')
